## Bert 模型的介绍

In [1]:
# 导入包
from transformers import BertForSequenceClassification, AutoTokenizer

/Users/chan/anaconda3/envs/rag_teach/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = BertForSequenceClassification.from_pretrained("../rag_qa/core/bert_query_classifier")

In [3]:
model
# text = "中国的首都是[MASK]京" --> MLM --> [MASK] 对应概率分布,取概率最大的是"北"

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
# 加载模型和分词器
model_path = "../rag_qa/core/bert_query_classifier"
# model = BertForSequenceClassification.from_pretrained(model_path)  # 加载模型
tokenizer = AutoTokenizer.from_pretrained(model_path)


In [5]:
# tokenizer.save_pretrained(".")

In [12]:
# 输出bert 模型的架构
# BERT->pooler->classifier
model.bert

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [6]:
# 单个句子编码
text = "黑马的Java课程安排有哪些？"
inputs = tokenizer(text, return_tensors="pt") # 返回pytorch
inputs

{'input_ids': tensor([[ 101, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,  763,
         8043,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [7]:
# 句子对编码
texts = ["黑马的Java课程安排有哪些？", "黑马的Python课程安排有哪些？"]
inputs = tokenizer(texts[0], texts[1], return_tensors="pt")
inputs

{'input_ids': tensor([[ 101, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,  763,
         8043,  102, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,
          763, 8043,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}

In [7]:
texts = ["黑马的Java课程安排有哪些？", "黑马的Python有哪些？"]
tokenizer(texts, return_tensors="pt", padding=True)


{'input_ids': tensor([[ 101, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,  763,
         8043,  102],
        [ 101, 7946, 7716, 4638,  100, 3300, 1525,  763, 8043,  102,    0,    0,
            0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}

In [10]:
# 测试模型输出
text = "详细介绍下JAVA的JDK" # 意图分类 专业知识
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs) # **表示将字典中的键值对作为参数传递给函数
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[ 6.5547, -6.7713]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [11]:
# 获取预测结果
outputs.logits.argmax(dim=-1)

tensor([0])